In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Hp\\Desktop\\cifar10-image-classificatio\\research'

In [3]:
os.chdir("../")

In [4]:
import numpy as np
from pathlib import Path

In [5]:
%pwd

'c:\\Users\\Hp\\Desktop\\cifar10-image-classificatio'

In [6]:
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple


@dataclass
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path

    epochs: int
    batch_size: int
    is_augmentation: bool
    image_size: Tuple[int, int, int]

    learning_rate: float
    weight_decay: float

    rotation_range: int
    width_shift_range: float
    height_shift_range: float
    horizontal_flip: bool
    zoom_range: float
    brightness_range: tuple
    shear_range: float
    channel_shift_range: float

    reduce_lr_factor: float
    reduce_lr_patience: int
    min_learning_rate: float
    early_stopping_patience: int

In [7]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [8]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [9]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories

from pathlib import Path


class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self):

        training = self.config.training

        # ✅ FIXED KEY (prepare_model not prepare_base_model)
        prepare_model = self.config.prepare_model

        params = self.params

        create_directories([Path(training.root_dir)])

        training_data = Path("artifacts/data_ingestion")

        return TrainingConfig(

            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),

            updated_base_model_path=Path(prepare_model.model_path),

            training_data=training_data,

            epochs=int(params.EPOCHS),
            batch_size=int(params.BATCH_SIZE),
            image_size=tuple(params.IMAGE_SIZE),

            is_augmentation=str(params.AUGMENTATION).lower() == "true",

            learning_rate=float(params.LEARNING_RATE),
            weight_decay=float(params.WEIGHT_DECAY),

            rotation_range=int(params.ROTATION_RANGE),
            width_shift_range=float(params.WIDTH_SHIFT_RANGE),
            height_shift_range=float(params.HEIGHT_SHIFT_RANGE),

            horizontal_flip=str(params.HORIZONTAL_FLIP).lower() == "true",
            zoom_range=float(params.ZOOM_RANGE),
            brightness_range=tuple(params.BRIGHTNESS_RANGE),
            shear_range=float(params.SHEAR_RANGE),
            channel_shift_range=float(params.CHANNEL_SHIFT_RANGE),

            reduce_lr_factor=float(params.REDUCE_LR_FACTOR),
            reduce_lr_patience=int(params.REDUCE_LR_PATIENCE),
            min_learning_rate=float(params.MIN_LEARNING_RATE),
            early_stopping_patience=int(params.EARLY_STOPPING_PATIENCE)
        )

In [10]:
import tensorflow as tf
import numpy as np
from pathlib import Path


class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def load_data(self):

        data_path = Path(self.config.training_data)

        self.X_train = np.load(data_path / "X_train.npy")
        self.X_test = np.load(data_path / "X_test.npy")
        self.y_train = np.load(data_path / "y_train.npy")
        self.y_test = np.load(data_path / "y_test.npy")

    def get_base_model(self):

        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def get_callbacks(self):

        return [
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=self.config.reduce_lr_factor,
                patience=self.config.reduce_lr_patience,
                min_lr=self.config.min_learning_rate,
                verbose=1
            ),

            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=self.config.early_stopping_patience,
                restore_best_weights=True,
                verbose=1
            )
        ]

    def train(self):

        self.load_data()
        self.get_base_model()

        optimizer = tf.keras.optimizers.Adam(
            learning_rate=self.config.learning_rate
        )

        self.model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",  # ✅ FIX
            metrics=["accuracy"]
        )

        history = self.model.fit(
            self.X_train,
            self.y_train,
            validation_data=(self.X_test, self.y_test),
            epochs=self.config.epochs,
            batch_size=self.config.batch_size,
            callbacks=self.get_callbacks(),
            verbose=2
        )

        self.model.save(self.config.trained_model_path)

        return history

In [11]:
try:
    config_manager = ConfigurationManager(
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    )

    training_config = config_manager.get_training_config()

    training = Training(training_config)

    training.train()

    print("✅ Training completed successfully")

except Exception as e:
    print("❌ Error:")
    raise e

[2026-07-01 19:41:46,757: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-01 19:41:46,767: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-01 19:41:46,773: INFO: common: created directory at: artifacts]
[2026-07-01 19:41:46,777: INFO: common: created directory at: artifacts\training]
Epoch 1/25
782/782 - 1045s - loss: 2.2624 - accuracy: 0.3872 - val_loss: 1.5134 - val_accuracy: 0.5077 - lr: 5.0000e-04 - 1045s/epoch - 1s/step
Epoch 2/25
782/782 - 709s - loss: 1.6240 - accuracy: 0.5404 - val_loss: 1.1982 - val_accuracy: 0.6140 - lr: 5.0000e-04 - 709s/epoch - 907ms/step
Epoch 3/25
782/782 - 614s - loss: 1.2255 - accuracy: 0.6243 - val_loss: 0.9713 - val_accuracy: 0.6842 - lr: 5.0000e-04 - 614s/epoch - 785ms/step
Epoch 4/25
782/782 - 783s - loss: 1.0622 - accuracy: 0.6770 - val_loss: 0.9508 - val_accuracy: 0.7124 - lr: 5.0000e-04 - 783s/epoch - 1s/step
Epoch 5/25
782/782 - 784s - loss: 0.9336 - accuracy: 0.7196 - val_loss: 0.8473 - val_accuracy:

c:\Users\Hp\anaconda3\envs\abc\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


✅ Training completed successfully


  Debugging

In [12]:
from tensorflow.keras.models import load_model

model = load_model(r"C:\Users\Hp\Desktop\cifar10-image-classificatio\model\model.h5")

print(model.input_shape)
print(model.output_shape)

(None, 32, 32, 3)
(None, 10)


In [13]:
import os
import time

path = r"C:\Users\Hp\Desktop\cifar10-image-classificatio\model\model.h5"

print("Exists:", os.path.exists(path))
print("Size (MB):", os.path.getsize(path)/1024/1024)
print("Modified:", time.ctime(os.path.getmtime(path)))

Exists: True
Size (MB): 13.711799621582031
Modified: Wed Jul  1 16:36:22 2026


In [15]:
training = Training(training_config)

history = training.train()

Epoch 1/25
782/782 - 683s - loss: 2.2613 - accuracy: 0.3866 - val_loss: 1.5767 - val_accuracy: 0.5122 - lr: 5.0000e-04 - 683s/epoch - 874ms/step
Epoch 2/25
782/782 - 652s - loss: 1.5788 - accuracy: 0.5541 - val_loss: 1.5236 - val_accuracy: 0.5783 - lr: 5.0000e-04 - 652s/epoch - 833ms/step
Epoch 3/25
782/782 - 1554s - loss: 1.2133 - accuracy: 0.6262 - val_loss: 1.1283 - val_accuracy: 0.6531 - lr: 5.0000e-04 - 1554s/epoch - 2s/step
Epoch 4/25
782/782 - 686s - loss: 1.0489 - accuracy: 0.6821 - val_loss: 1.0873 - val_accuracy: 0.7084 - lr: 5.0000e-04 - 686s/epoch - 877ms/step
Epoch 5/25
782/782 - 661s - loss: 0.9331 - accuracy: 0.7169 - val_loss: 0.8148 - val_accuracy: 0.7531 - lr: 5.0000e-04 - 661s/epoch - 845ms/step
Epoch 6/25
782/782 - 670s - loss: 0.8486 - accuracy: 0.7449 - val_loss: 0.8314 - val_accuracy: 0.7520 - lr: 5.0000e-04 - 670s/epoch - 857ms/step
Epoch 7/25
782/782 - 655s - loss: 0.7839 - accuracy: 0.7721 - val_loss: 0.7970 - val_accuracy: 0.7724 - lr: 5.0000e-04 - 655s/epoch

In [17]:
import pickle
import json
from pathlib import Path
import numpy as np

history_dir = Path("artifacts/training")
history_dir.mkdir(parents=True, exist_ok=True)

# Save as Pickle
with open(history_dir / "history.pkl", "wb") as f:
    pickle.dump(history.history, f)

# Convert NumPy data types to Python native types
history_json = {
    key: [float(x) if isinstance(x, (np.floating, np.float32, np.float64)) else x for x in values]
    for key, values in history.history.items()
}

# Save as JSON
with open(history_dir / "history.json", "w") as f:
    json.dump(history_json, f, indent=4)

print("History saved successfully!")

History saved successfully!
